# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [69]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [70]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_files = list(Path(pdf_directory).glob('**/*.pdf'))

    for pdf_file in pdf_files:
        documents = PyPDFLoader(str(pdf_file)).load()
        for doc in documents:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'
        all_documents.extend(documents)

    return all_documents
all_pdf_documents = process_all_pdfs("data/pdfs")


In [71]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 6.0.1 (Windows)', 'creator': 'Acrobat PDFMaker 6.0 for Word', 'creationdate': '2004-09-02T13:13:30-05:00', 'moddate': '2004-09-02T13:15:39-05:00', 'title': 'Microsoft Word - X21.doc', 'author': 'Richard Williams', 'company': 'University of Notre Dame', 'sourcemodified': 'D:20040902181305', 'source': 'data/pdfs/x21.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'x21.pdf', 'file_type': 'pdf'}, page_content='Normal distribution \n \nThe normal distribution is the most widely known and used of all distributions.  Because the \nnormal distribution approximates many natural phenomena so well, it has developed into a \nstandard of reference for many probability problems. \n \n \n \nI. Characteristics of the Normal distribution \n \n• Symmetric, bell shaped \n• Continuous for all values of X between -∞ and ∞ so that each conceivable interval of real \nnumbers has a probability other than zero. \n• -∞ ≤ X ≤ ∞ \n• Two para

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [72]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [73]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs
chunks=split_documents(all_pdf_documents)
chunks

Split 10 documents into 23 chunks


[Document(metadata={'producer': 'Acrobat Distiller 6.0.1 (Windows)', 'creator': 'Acrobat PDFMaker 6.0 for Word', 'creationdate': '2004-09-02T13:13:30-05:00', 'moddate': '2004-09-02T13:15:39-05:00', 'title': 'Microsoft Word - X21.doc', 'author': 'Richard Williams', 'company': 'University of Notre Dame', 'sourcemodified': 'D:20040902181305', 'source': 'data/pdfs/x21.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'x21.pdf', 'file_type': 'pdf'}, page_content='Normal distribution \n \nThe normal distribution is the most widely known and used of all distributions.  Because the \nnormal distribution approximates many natural phenomena so well, it has developed into a \nstandard of reference for many probability problems. \n \n \n \nI. Characteristics of the Normal distribution \n \n• Symmetric, bell shaped \n• Continuous for all values of X between -∞ and ∞ so that each conceivable interval of real \nnumbers has a probability other than zero. \n• -∞ ≤ X ≤ ∞ \n• Two para

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [74]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [75]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return np.array(embeddings)



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [76]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        ids, metadatas, contents, embeddings_list = [], [], [], []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(uuid.uuid4().hex[:8])
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            contents.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=contents
        )

In [77]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 6.0.1 (Windows)', 'creator': 'Acrobat PDFMaker 6.0 for Word', 'creationdate': '2004-09-02T13:13:30-05:00', 'moddate': '2004-09-02T13:15:39-05:00', 'title': 'Microsoft Word - X21.doc', 'author': 'Richard Williams', 'company': 'University of Notre Dame', 'sourcemodified': 'D:20040902181305', 'source': 'data/pdfs/x21.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'x21.pdf', 'file_type': 'pdf'}, page_content='Normal distribution \n \nThe normal distribution is the most widely known and used of all distributions.  Because the \nnormal distribution approximates many natural phenomena so well, it has developed into a \nstandard of reference for many probability problems. \n \n \n \nI. Characteristics of the Normal distribution \n \n• Symmetric, bell shaped \n• Continuous for all values of X between -∞ and ∞ so that each conceivable interval of real \nnumbers has a probability other than zero. \n• -∞ ≤ X ≤ ∞ \n• Two para

## convert text to embeddings


In [78]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['Normal distribution \n \nThe normal distribution is the most widely known and used of all distributions.  Because the \nnormal distribution approximates many natural phenomena so well, it has developed into a \nstandard of reference for many probability problems. \n \n \n \nI. Characteristics of the Normal distribution \n \n• Symmetric, bell shaped \n• Continuous for all values of X between -∞ and ∞ so that each conceivable interval of real \nnumbers has a probability other than zero. \n• -∞ ≤ X ≤ ∞ \n• Two parameters, µ and σ.  Note that the normal distribution is actually a family of \ndistributions, since µ and σ determine the shape of the distribution. \n \n• The rule for a normal density function is \ne2\n1 = ) , f(x;\n22 /2)--(x\n2\n2 σµ\nσπσµ  \n• The notation N(µ, σ2) means normally distributed with mean µ and variance σ2.  If we say \nX ∼ N(µ, σ2) we mean that X is distributed N(µ, σ2). \n• About 2/3 of all cases fall within one standard deviation of the mean, that is \n \nP

In [79]:

embedding_manager = EmbeddingManager()
vectorstore = VectorStore()
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6818.93it/s]
/var/folders/57/x_zzd1m115jf02y44k7knyhc0000gn/T/ipykernel_11866/1290750164.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Model loaded successfully. Embedding dimension: 384


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.52it/s]


# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [80]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embeddings([query])

        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )

        retrieved_docs = []
        for rank, (doc_id, content, metadata, distance) in enumerate(
            zip(results['ids'][0], results['documents'][0],
                results['metadatas'][0], results['distances'][0]), start=1
        ):
            similarity_score = 1 - distance
            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    'id': doc_id,
                    'content': content,
                    'metadata': metadata,
                    'similarity_score': similarity_score,
                    'distance': distance,
                    'rank': rank
                })

        return retrieved_docs
    


In [81]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve(" put something here similar to your data to retrieve")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]


[]

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [83]:
def rag_simple(query, retriever, llm, top_k=3):
    docs = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([d['content'] for d in docs])

    prompt = f"""Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content

In [84]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

# Initialize the LLM using the current Llama 3.3 model
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile", 
    temperature=0
)

In [92]:
answer=rag_simple("Why is the normal distribution useful?",rag_retriever,llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 23.32it/s]


The normal distribution is useful for three main reasons: 

1. Many things are actually normally distributed, or very close to it, such as height and intelligence.
2. The normal distribution is easy to work with mathematically, and methods developed using normal theory often work well even when the distribution is not normal.
3. There is a strong connection between the size of a sample and the extent to which a sampling distribution approaches the normal form, allowing for approximation of many sampling distributions based on large samples.


# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [86]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    docs = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    context = "\n\n".join([d['content'] for d in docs])

    sources = []
    for d in docs:
        sources.append({
            'source_file': d['metadata'].get('source_file', 'unknown'),
            'page': d['metadata'].get('page', 'unknown'),
            'similarity_score': d['similarity_score'],
            'content_preview': d['content'][:150] + "..."
        })

    confidence = max([d['similarity_score'] for d in docs], default=0.0)

    prompt = f"""Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)

    result = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        result['context'] = context

    return result


In [87]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        docs = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        sources = []
        for d in docs:
            sources.append({
                'source_file': d['metadata'].get('source_file', 'unknown'),
                'page': d['metadata'].get('page', 'unknown'),
                'similarity_score': d['similarity_score'],
                'content_preview': d['content'][:150] + "..."
            })

        context = "\n\n".join([d['content'] for d in docs])

        prompt = f"""Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:"""

        if stream:
            print("Streaming prompt:")
            for i in range(0, len(prompt), 50):
                print(prompt[i:i+50], end="", flush=True)
                time.sleep(0.05)
            print()

        response = self.llm.invoke(prompt)
        answer = response.content

        citations = "\n\nSources:\n"
        for i, s in enumerate(sources, start=1):
            citations += f"[{i}] {s['source_file']} (page {s['page']}, score: {s['similarity_score']:.2f})\n"

        answer_with_citations = answer + citations

        summary = None
        if summarize:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n\n{answer}"
            summary = self.llm.invoke(summary_prompt).content

        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }